### Data Loading

In [1]:
import pandas as pd

buzzfeed_real = pd.read_csv('data/buzzfeed_real_news_content.csv')
buzzfeed_fake = pd.read_csv('data/buzzfeed_fake_news_content.csv')

In [2]:
buzzfeed_real.head(1)

,id,title,text,url,top_img,authors,source,publish_date,movies,images,canonical_link,meta_data
0,Real_1-Webpage,Another Terrorist Attack in NYC…Why Are we STI...,"On Saturday, September 17 at 8:30 pm EST, an e...",http://eaglerising.com/36942/another-terrorist...,http://eaglerising.com/wp-content/uploads/2016...,"View All Posts,Leonora Cravotta",http://eaglerising.com,{'$date': 1474528230000},NaN,http://constitution.com/wp-content/uploads/201...,http://eaglerising.com/36942/another-terrorist...,"{""description"": ""\u201cWe believe at this poin..."


In [3]:
buzzfeed_fake.head(1)

,id,title,text,url,top_img,authors,source,publish_date,movies,images,canonical_link,meta_data
0,Fake_1-Webpage,Proof The Mainstream Media Is Manipulating The...,I woke up this morning to find a variation of ...,http://www.addictinginfo.org/2016/09/19/proof-...,http://addictinginfo.addictinginfoent.netdna-c...,Wendy Gittleson,http://www.addictinginfo.org,{'$date': 1474243200000},NaN,"http://i.imgur.com/JeqZLhj.png,http://addictin...",http://addictinginfo.com/2016/09/19/proof-the-...,"{""publisher"": ""Addicting Info | The Knowledge ..."


### Preprocessing

In [4]:
#function to be applied to article titles and contents for initial text cleaning
import re
import string

def clean_text(text):
    
    #type checking
    if not isinstance(text, str):
        return []

    #lowercasing
    text = text.lower()
    
    #removing punctuation
    text = re.sub('\[.*?\]', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\w*\d\w*', '', text)

    text = re.sub('[’‘’“”…]', '', text)
    text = re.sub('\n', '', text)
    text = ' '.join(re.findall(r'\b[a-zA-Z0-9]+\b', text))


    return text

In [5]:
#fake news preprocessing
buzzfeed_fake['clean_title'] = buzzfeed_fake['title'].apply(clean_text)
buzzfeed_fake['clean_text'] = buzzfeed_fake['text'].apply(clean_text)

#real news preprocessing
buzzfeed_real['clean_title'] = buzzfeed_real['title'].apply(clean_text)
buzzfeed_real['clean_text'] = buzzfeed_real['text'].apply(clean_text)

In [6]:
buzzfeed_fake.head(1)

,id,title,text,url,top_img,authors,source,publish_date,movies,images,canonical_link,meta_data,clean_title,clean_text
0,Fake_1-Webpage,Proof The Mainstream Media Is Manipulating The...,I woke up this morning to find a variation of ...,http://www.addictinginfo.org/2016/09/19/proof-...,http://addictinginfo.addictinginfoent.netdna-c...,Wendy Gittleson,http://www.addictinginfo.org,{'$date': 1474243200000},NaN,"http://i.imgur.com/JeqZLhj.png,http://addictin...",http://addictinginfo.com/2016/09/19/proof-the-...,"{""publisher"": ""Addicting Info | The Knowledge ...",proof the mainstream media is manipulating the...,i woke up this morning to find a variation of ...


In [7]:
buzzfeed_real.head(1)

,id,title,text,url,top_img,authors,source,publish_date,movies,images,canonical_link,meta_data,clean_title,clean_text
0,Real_1-Webpage,Another Terrorist Attack in NYC…Why Are we STI...,"On Saturday, September 17 at 8:30 pm EST, an e...",http://eaglerising.com/36942/another-terrorist...,http://eaglerising.com/wp-content/uploads/2016...,"View All Posts,Leonora Cravotta",http://eaglerising.com,{'$date': 1474528230000},NaN,http://constitution.com/wp-content/uploads/201...,http://eaglerising.com/36942/another-terrorist...,"{""description"": ""\u201cWe believe at this poin...",another terrorist attack in nycwhy are we stil...,on saturday september at pm est an explosion r...


In [8]:
#set up data for train test split
buzzfeed_fake['label'] = 0
buzzfeed_real['label'] = 1

#create single dataframe and shuffle it up
full_df = pd.concat([buzzfeed_fake, buzzfeed_real], axis=0, ignore_index = True).sample(frac = 1, random_state = 101).reset_index(drop = True)

### Tokenizing

In [ ]:
#create a document term matrix
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(stop_words = 'english')
df_cv = cv.fit_transform(full_df['clean_text'])

df_dtm = pd.DataFrame(df_cv.toarray(), columns = cv.get_feature_names_out(), index = full_df.index)

df_dtm

,aaron,ab,abandon,abandoned,abandoning,abandonment,abbi,abbreviated,abc,abcs,...,zach,zapfelkristen,zero,zerosum,zimmer,zingers,zipf,zipper,zone,zoomedin
0,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
177,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
178,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
179,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
180,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [10]:
labelled_dtm = pd.concat([df_dtm, full_df['label']], axis = 1)

In [ ]:
#save pkl for model training
pd.to_pickle(labelled_dtm, 'data/labelled_dtm.pkl')